# Deploy - Terraform + Docker Deployment Agent

In this exercise, we build an agent that writes a Terraform configuration from scratch and uses it to deploy a Docker container on the local Docker daemon.

The agent iterates between `terraform_validate` / `terraform_plan` and fixes the configuration until the infrastructure can be applied successfully.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


**Prerequisites:** Terraform and Docker must be installed on your machine and the Docker daemon must be running. The agent uses the [kreuzwerker/docker](https://registry.terraform.io/providers/kreuzwerker/docker) provider, which talks to the local Docker socket by default.

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
# Folder where the generated Terraform files will live.
terraform_folder = pathlib.Path("/tmp/dlm03-deploy/")
terraform_folder.mkdir(parents=True, exist_ok=True)

# The Docker image to deploy. The agent will spin up a container for it via Terraform.
docker_image = "nginx:1.27-alpine"
container_name = "dlm03-nginx"
host_port = 8080
container_port = 80

# MCP Server

The server exposes file tools scoped to the Terraform working directory and a set of `terraform_*` tools wrapping the CLI.

In [ ]:
import shlex
import subprocess

from mcp.server import FastMCP

SERVER = FastMCP()

TERRAFORM_FOLDER = terraform_folder.resolve()


@SERVER.tool()
def list_terraform_files() -> list[str]:
    """List the files in `path` that match the given glob expression, returning absolute paths.

    Prefer narrow globs (e.g. `*.py`, `requirements*.txt`) to avoid very large listings.
    """
    root = TERRAFORM_FOLDER
    files = [p for p in root.glob(glob) if p.is_file()]
    if not files:
        return "No files found."
    return [str(p) for p in root.glob("*") if p.is_file() and ".git" not in str(p)]

@SERVER.tool()
def read_terraform_file(filename: str) -> str:
    """Read the content of a file inside the Terraform working directory."""
    path = TERRAFORM_FOLDER / filename
    if not path.exists():
        return "File not found."
    return path.read_text()


@SERVER.tool()
def write_terraform_file(filename: str, content: str) -> str:
    """Write a Terraform file into the working directory.

    `filename` must be a simple name (no path traversal).
    """
    if "/" in filename or ".." in filename:
        return "filename must be a simple name inside the Terraform working directory."
    path = TERRAFORM_FOLDER / filename
    path.write_text(content)
    return f"Wrote {path}"


def _run_terraform(args: list[str], timeout: int = 300) -> str:
    try:
        result = subprocess.run(
            ["terraform", *args],
            cwd=TERRAFORM_FOLDER,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
    except FileNotFoundError:
        return "`terraform` CLI not found on PATH."
    except subprocess.TimeoutExpired:
        return f"terraform {' '.join(args)} timed out after {timeout}s."
    output = (result.stdout or "") + "\n" + (result.stderr or "")
    return f"exit_code={result.returncode}\n{output[-5000:]}"


@SERVER.tool()
def terraform_init() -> str:
    """Run `terraform init` in the working directory."""
    return _run_terraform(["init", "-input=false", "-no-color"])


@SERVER.tool()
def terraform_validate() -> str:
    """Run `terraform validate`."""
    return _run_terraform(["validate", "-no-color"])


@SERVER.tool()
def terraform_plan() -> str:
    """Run `terraform plan`."""
    return _run_terraform(["plan", "-input=false", "-no-color"])


@SERVER.tool()
def terraform_apply() -> str:
    """Run `terraform apply -auto-approve` (deploys the container)."""
    return _run_terraform(["apply", "-auto-approve", "-input=false", "-no-color"], timeout=600)


@SERVER.tool()
def terraform_destroy() -> str:
    """Tear down the deployed resources."""
    return _run_terraform(["destroy", "-auto-approve", "-input=false", "-no-color"])


@SERVER.tool()
def docker_ps() -> str:
    """List running Docker containers - useful to verify the deployment."""
    try:
        result = subprocess.run(
            ["docker", "ps", "--format", "{{.Names}}\t{{.Image}}\t{{.Status}}\t{{.Ports}}"],
            capture_output=True, text=True, timeout=30,
        )
    except FileNotFoundError:
        return "`docker` CLI not found on PATH."
    return (result.stdout or "") + (result.stderr or "")

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
    "log_level": "warning",
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS, daemon=True)
MCP_THREAD.start()

MCP_URL = f"http://{HOST}:{PORT}/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are an expert DevOps engineer specialized in Infrastructure-as-Code.

Your goal is to deploy a Docker container using Terraform (with the
[kreuzwerker/docker](https://registry.terraform.io/providers/kreuzwerker/docker) provider)
against the local Docker daemon.

Workflow:
1. Write a minimal, self-contained Terraform configuration in the working directory:
   - `main.tf` declaring the `kreuzwerker/docker` provider and the desired `docker_image` /
     `docker_container` resources.
   - `variables.tf` / `outputs.tf` if it keeps things cleaner.
   The container should publish the requested host port to the requested container port.
2. Run `terraform_init`, then `terraform_validate`. Fix any errors and repeat until both
   succeed.
3. Run `terraform_plan` and read the output - sanity-check that the plan matches intent.
4. Run `terraform_apply` to deploy.
5. Verify with `docker_ps` that the container is up.

Always `terraform_init` before `plan` / `apply`. Stop once the deployment is verified.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please write the Terraform configuration needed to deploy `{{ docker_image }}` as a
container named `{{ container_name }}`, publishing host port {{ host_port }} to container
port {{ container_port }}, then deploy it and verify it is running."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
user_message = user_message_template.render(
    docker_image=docker_image,
    container_name=container_name,
    host_port=host_port,
    container_port=container_port,
)

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(MCP_URL) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)

## Teardown

Once you're done, tear down the deployed container to keep the local Docker daemon clean.

async with tools_lib.mcp_session(MCP_URL) as session:
    result = await session.call_tool("terraform_destroy", {})
    for content in result.content:
        print(getattr(content, "text", content))